In [34]:
import os 
import requests
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import display,Markdown,update_display
from bs4 import BeautifulSoup
import json
import logging

In [35]:
load_dotenv()

True

In [36]:
logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s | %(message)s"
)
logger = logging.getLogger(__name__)


In [37]:
openai = OpenAI(
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    api_key=os.getenv("GEMINI_API_KEY")
)

In [38]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

In [39]:
from urllib.parse import urljoin

def fectch_websiite_links(url):
    response = requests.get(url, headers=headers)
    response.raise_for_status()

    soup = BeautifulSoup(response.content, "html.parser")

    links = []

    for tag in soup.find_all("a"):
        href = tag.get("href")

        if href:
            href = urljoin(url, href)

            if not (
                href.startswith("mailto:")
                or href.startswith("tel:")
                or href.startswith("javascript:")
                or "#" in href
            ):
                links.append(href)

    return list(set(links))

In [40]:
def fetch_website_contents(url):
    response = requests.get(url, headers=headers)
    response.raise_for_status()

    soup = BeautifulSoup(response.content, "html.parser")

    for tag in soup(["script", "style", "noscript", "svg", "img"]):
        tag.decompose()

    text = soup.get_text(separator="\n", strip=True)

    return text

In [41]:
link_system_prompt = """
You are an expert website navigator.

Your task is to analyze a list of URLs extracted from a company's website and identify ONLY the pages that contain important information for generating a professional company brochure.

Return ONLY a valid JSON object.

Rules:
- Only include links that are likely to contain useful company information.
- Always return absolute URLs.
- Do not invent or modify URLs.
- Do not include duplicate URLs.
- Ignore anchors (#), mailto links, tel links, javascript links, PDFs, images, videos, login pages, cart pages, search pages, privacy/legal pages, cookie pages, terms pages, or social media links unless they are the company's primary contact page.
- If multiple URLs point to the same section, keep the best one.
- If a page belongs to more than one category, choose the most relevant category.
- If no useful links exist, return:
{
  "links": []
}

Output format:

{
  "links": [
    {
      "type": "<category>",
      "url": "<absolute_url>"
    }
  ]
}

Example 1

Input:
[
  "https://acme.com/",
  "https://acme.com/about",
  "https://acme.com/products",
  "https://acme.com/blog",
  "https://acme.com/privacy",
  "https://twitter.com/acme"
]

Output:
{
  "links": [
    {
      "type": "homepage",
      "url": "https://acme.com/"
    },
    {
      "type": "about",
      "url": "https://acme.com/about"
    },
    {
      "type": "products",
      "url": "https://acme.com/products"
    },
    {
      "type": "blog",
      "url": "https://acme.com/blog"
    }
  ]
}

Return ONLY the JSON object.
"""

In [42]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company.
Respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.
Links (some might be relative links):
"""

    links = fectch_websiite_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [43]:
def select_relevent_links(url):

    response = openai.chat.completions.create(
        model="gemini-3.1-flash-lite",
        messages=[
            {
                "role": "system",
                "content": link_system_prompt
            },
            {
                "role": "user",
                "content": get_links_user_prompt(url)
            }
        ],
        response_format={
            "type": "json_object"
        }
    )

    result = response.choices[0].message.content

    result = result.replace("```json", "")
    result = result.replace("```", "")
    result = result.strip()

    return json.loads(result)

In [44]:
def fetch_page_and_all_relevant_links(url):

    contents = fetch_website_contents(url)

    relevant_links = select_relevent_links(url)

    result = f"## Landing Page\n\n{contents}\n\n"

    result += "## Relevant Pages\n"

    for link in relevant_links["links"]:

        result += f"\n\n### {link['type']}\n"

        try:
            result += fetch_website_contents(link["url"])
        except Exception:
            pass

    return result

In [45]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

In [46]:
def get_brochure_user_prompt(company_name, url):

    user_prompt = f"""
You are looking at a company called {company_name}.

Here are the contents of its landing page and other relevant pages.

Create a professional brochure in Markdown.

"""

    user_prompt += fetch_page_and_all_relevant_links(url)

    return user_prompt[:5000]

In [47]:
def create_brochure(company_name, url):

    response = openai.chat.completions.create(
        model="gemini-3.1-flash-lite",
        messages=[
            {
                "role": "system",
                "content": brochure_system_prompt
            },
            {
                "role": "user",
                "content": get_brochure_user_prompt(company_name, url)
            }
        ]
    )

    display(Markdown(response.choices[0].message.content))

In [48]:
def stream_brochure(company_name, url):

    stream = openai.chat.completions.create(
        model="gemini-3.1-flash-lite",
        messages=[
            {
                "role": "system",
                "content": brochure_system_prompt
            },
            {
                "role": "user",
                "content": get_brochure_user_prompt(company_name, url)
            }
        ],
        stream=True
    )

    response = ""

    handle = display(Markdown(""), display_id=True)

    for chunk in stream:

        delta = chunk.choices[0].delta.content

        if delta is not None:
            response += delta

            update_display(
                Markdown(response),
                display_id=handle.display_id
            )

In [49]:
stream_brochure(
    "HuggingFace",
    "https://huggingface.co"
)

INFO | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
INFO | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"


# Hugging Face: The AI Community Building the Future

## Who We Are
Hugging Face is the premier collaboration platform for the global machine learning community. We are building the foundational infrastructure for the next generation of AI, acting as the central hub where researchers, developers, and organizations discover, share, and build state-of-the-art machine learning models, datasets, and applications. 

By prioritizing open-source values and community-driven development, we empower users to move faster and build more effectively, whether they are working with text, image, video, audio, or 3D modalities.

## Our Impact
We are the "Home of Machine Learning," hosting a vast ecosystem that includes:
* **Over 2 Million Models:** A massive library of pre-trained models accessible to everyone.
* **Over 500,000 Datasets:** The data resources necessary to train and fine-tune next-generation AI.
* **1 Million+ AI Applications:** Interactive tools and demos hosted on our Spaces platform.
* **Global Collaboration:** Over 50,000 organizations—including industry leaders like Meta, Google, Microsoft, Amazon, Intel, and AI2—utilize Hugging Face to house their models and foster innovation.

## Our Technology Stack
We provide the open-source tools that power modern AI development, including:
* **Transformers & Diffusers:** Industry-standard libraries for state-of-the-art models.
* **Tokenizers & PEFT:** Highly optimized tools for performance and efficient fine-tuning.
* **smolagents:** Our modern framework for building powerful AI agents.
* **Transformers.js:** Enabling state-of-the-art ML to run directly in the browser.

## Enterprise Solutions
For businesses, Hugging Face offers enterprise-grade infrastructure that balances innovation with control. Our platform provides:
* **Security & Governance:** Single Sign-On, audit logs, and resource groups.
* **Scalable Infrastructure:** Optimized inference endpoints, dedicated support, and private dataset management.
* **Unified API:** Access to 45,000+ models from leading AI providers through a single, seamless interface.

## Join the Future
Hugging Face is built by a community of passionate builders. We are committed to transparency, open science, and accelerating the reach of AI to every corner of the globe.

### Careers & Culture
We are looking for individuals who are excited about the intersection of open source, machine learning, and human collaboration. At Hugging Face, you will work on the core libraries that are defining the future of how humanity interacts with intelligence. 

If you want to help us build the foundational tools that millions of developers use every day, we invite you to explore our career opportunities. 

**Connect with us:**
* **Join the community:** [huggingface.co](https://huggingface.co)
* **Engage:** Join our discussions on our Discord server or the community forum.
* **Collaborate:** Visit our GitHub repositories to contribute to our open-source stack.

**Hugging Face: Creating, discovering, and collaborating on ML, together.**